# Pitch your partition

Everyone in this room is a node. Two people who sit close are joined by an edge. Your group makes **one** partition of these nodes and has three minutes to say why it is the one to use. There is no answer key.

Run *Setup*, then *The graph*. Then go to your group's section.

In [ ]:
#@title Setup. Press the play button. Nothing to type here. { display-mode: "form" }
try:
    import igraph
except ImportError:
    %pip install -q python-igraph
    import igraph

from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

SEATS_URL = "https://raw.githubusercontent.com/skojaku/adv-net-sci/claude/network-robustness-student-project-wwb19j/lecture-note/m05-clustering/pitch/seats.csv"
GRAPH, K, EPS = "eps", 3, None   # set by the lecturer: "knn" (k nearest) or "eps" (within EPS)

try:
    seats = pd.read_csv(SEATS_URL)
except Exception:
    seats = pd.read_csv(SEATS_URL.replace("seats.csv", "seats-example.csv"))
    print("#" * 70)
    print("REHEARSAL DATA: seats.csv is not published yet, so this is an invented room.")
    print("Run this notebook again once the lecturer says the seats are up.")
    print("#" * 70)

P = seats[["x", "y"]].to_numpy(dtype=float)
N = len(P)
D = np.linalg.norm(P[:, None, :] - P[None, :, :], axis=2)
np.fill_diagonal(D, np.inf)
if GRAPH == "knn":
    EDGES = sorted({tuple(sorted((i, int(j)))) for i in range(N) for j in np.argsort(D[i], kind="stable")[:K]})
else:
    eps = D.min(axis=1).max() if EPS is None else EPS      # None: the smallest value that leaves nobody alone
    EDGES = [(i, j) for i, j in combinations(range(N), 2) if D[i, j] <= eps + 1e-9]
graph = igraph.Graph(n=N, edges=EDGES)
A = np.array(graph.get_adjacency().data)

PALETTE = ["#3959A6", "#B14434", "#DAB167", "#6B6B6B", "#7A5195", "#E377C2", "#8C564B", "#6BAED6", "#1A1A1A", "#BDB8AA"]


def show(labels, matrix=True):
    """Draw the partition: colour = group, dashed = edge between groups. Then print the
    matrix: the diagonal is the number of edges inside a group, the rest is the number of
    edges between two groups."""
    labels = list(labels)
    assert len(labels) == N, f"labels needs one number per node: {N} numbers, you gave {len(labels)}"
    names = sorted(set(labels))
    fig, ax = plt.subplots(figsize=(10, 4.2))
    for u, v in EDGES:
        cross = labels[u] != labels[v]
        ax.plot(P[[u, v], 0], P[[u, v], 1], color="#1A1A1A" if cross else "#BDB8AA", lw=1.2 if cross else 1.8,
                ls=(0, (4, 3)) if cross else "-", zorder=1)
    colour = [PALETTE[names.index(g) % len(PALETTE)] for g in labels]
    ax.scatter(P[:, 0], P[:, 1], s=420, c=colour, edgecolor="#1A1A1A", lw=1, zorder=2)
    for i in range(N):
        ax.text(P[i, 0], P[i, 1], str(i), ha="center", va="center", fontsize=10, fontweight="bold",
                color="#1A1A1A" if colour[i] in ("#DAB167", "#6BAED6", "#BDB8AA") else "white", zorder=3)
    ax.set_aspect("equal"); ax.axis("off"); ax.margins(0.08)
    plt.show()
    if matrix:
        M = pd.DataFrame(0, index=names, columns=names)
        for u, v in EDGES:
            M.loc[labels[u], labels[v]] += 1
            if labels[u] != labels[v]:
                M.loc[labels[v], labels[u]] += 1
        M.insert(0, "size", [labels.count(g) for g in names])
        M.index.name = "group"
        display(M)


def check(labels):
    """The values to compare with your hand calculation."""
    labels = list(labels)
    rows = []
    for g in sorted(set(labels)):
        S = [i for i in range(N) if labels[i] == g]
        inside = int(A[np.ix_(S, S)].sum()) // 2
        vol = int(A[S].sum())
        rows.append(dict(group=g, size=len(S), inside=inside,
                         density=round(inside / (len(S) * (len(S) - 1) / 2), 3) if len(S) > 1 else None,
                         fewest_inside=int(A[np.ix_(S, S)].sum(axis=1).min()),
                         leaving=vol - 2 * inside, volume=vol))
    t = pd.DataFrame(rows).set_index("group")
    display(t)
    print(f"ratio cut      = sum of leaving/size   = {sum(r.leaving / r.size for r in t.itertuples()):.4f}")
    print(f"normalized cut = sum of leaving/volume = {sum(r.leaving / r.volume for r in t.itertuples() if r.volume):.4f}")
    print(f"modularity Q   = {graph.modularity(labels):.4f}")


print(f"{N} people, numbered 0 to {N - 1}. {len(EDGES)} edges.")

## The graph
Each node has a number. `labels[i]` will be the group of node `i`.

In [ ]:
show([0] * N, matrix=False)

## Groups 1 and 2 · by eye
**Group 1:** find a k-core, a clique and a pseudo-clique. Give its members the number 1 and everyone else 0.
**Group 2:** split the nodes into groups so that the ratio cut and the normalized cut are small. One number per group.

Edit the list, run the cell, look. Work out the values by hand from the matrix below the picture, then run the check.

In [ ]:
labels = [0] * N   # replace by one number per node, e.g. [0, 0, 1, 1, 0, ...]
show(labels)

In [ ]:
check(labels)

## Group 3 · igraph
Type the code from your sheet into the cell below. The network is `graph`.

In [ ]:
# type the code from your sheet

## Group 4 · graph-tool
Run Step A, then type the code from your sheet into the cell below.

In [ ]:
#@title Step A. Group 4 only. Press the play button and wait. { display-mode: "form" }
# graph-tool is not on PyPI. It is installed from conda into a private folder, and graph_tool(fn)
# runs your function fn there. Inside fn, `gt` is graph-tool and `g` is the network.
import inspect, io, json, os, pathlib, platform, subprocess, tarfile, tempfile, textwrap, urllib.request

GT = pathlib.Path(tempfile.gettempdir()) / "graph-tool"
GT_PYTHON = GT / "env" / "bin" / "python"

if not GT_PYTHON.exists():
    kind = {"Linux": "linux-64",
            "Darwin": "osx-arm64" if platform.machine() == "arm64" else "osx-64"}.get(platform.system())
    if kind is None:
        raise SystemExit("graph-tool does not run on Windows. Open this notebook in Colab instead.")
    print("Installing graph-tool: one to three minutes. Please wait.")
    url = f"https://micro.mamba.pm/api/micromamba/{kind}/latest"
    tarfile.open(fileobj=io.BytesIO(urllib.request.urlopen(url).read()), mode="r:bz2").extract("bin/micromamba", GT)
    subprocess.run([GT / "bin" / "micromamba", "create", "-y", "-q", "-p", GT / "env", "-c", "conda-forge", "graph-tool"],
                   env={**os.environ, "MAMBA_ROOT_PREFIX": str(GT / "root")}, check=True)


def graph_tool(fn):
    with tempfile.TemporaryDirectory() as tmp:
        tmp = pathlib.Path(tmp)
        (tmp / "in.json").write_text(json.dumps(dict(n=N, edges=EDGES)))
        (tmp / "run.py").write_text(
            "import json\nimport graph_tool.all as gt\n"
            "gt.seed_rng(1); gt.openmp_set_num_threads(1)\n"
            "d = json.load(open('in.json'))\n"
            "g = gt.Graph(directed=False); g.add_vertex(d['n']); g.add_edge_list(d['edges'])\n\n"
            + textwrap.dedent(inspect.getsource(fn))
            + f"\n\njson.dump({fn.__name__}(g), open('out.json', 'w'))\n")
        done = subprocess.run([GT_PYTHON, "run.py"], cwd=tmp, capture_output=True, text=True)
        if done.returncode:
            raise RuntimeError(done.stderr[-1500:])
        return json.loads((tmp / "out.json").read_text())


print("graph-tool is ready.")

In [ ]:
# type the code from your sheet